# GA4 Funnel Analizi: İstatistik ve Lojistik Regresyon

Bu notebook projenin iki iş sorusunu cevaplıyor:

- **Soru 5:** Ücretli reklamla (Paid Search) gelenlerle organik aramayla (Organic Search) gelenler arasındaki dönüşüm farkı istatistiksel olarak anlamlı mı?
- **Soru 6:** İlk ziyaretteki hangi davranışlar, kullanıcının 7 gün içinde satın alma olasılığını artırıyor?

Veri, BigQuery'deki mart tablolarından doğrudan okunuyor (`pandas-gbq`). İlk çalıştırmada tarayıcıda Google hesabıyla giriş ve BigQuery erişim izni istenir.

En sonda Tableau için mart tablolarının CSV çıktıları `data/` klasörüne yazılıyor.

In [1]:
import numpy as np
import pandas as pd
import pandas_gbq
from pathlib import Path
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep,
    proportion_effectsize,
)
from statsmodels.stats.power import NormalIndPower

PROJECT_ID = "ga4-funnel-analytics-515610"
DATASET = f"{PROJECT_ID}.ga4_funnel"

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


def run_query(sql: str) -> pd.DataFrame:
    """Sorguyu BigQuery'de çalıştırıp sonucu DataFrame olarak döndürür."""
    return pandas_gbq.read_gbq(sql, project_id=PROJECT_ID, progress_bar_type=None)

---
## 1. Soru 5: Ücretli ve organik arasında dönüşüm farkı var mı?

`mart_channel_quality` tablosunda ücretli reklamın dönüşümü %1,53, organiğinki %1,52 çıkmıştı. Bu fark gerçek mi, şans eseri mi?

- **H0 (sıfır hipotezi):** İki kanalın kullanıcı dönüşüm oranı eşit.
- **H1:** Eşit değil (çift yönlü test).
- **Dönüşüm:** Kullanıcı bazlı = alıcı ÷ kullanıcı (karar günlüğü 4.9).
- **Anlamlılık düzeyi:** α = 0,05.

In [2]:
channels = run_query(f"""
SELECT
  channel_group,
  SUM(users)      AS users,
  SUM(purchasers) AS purchasers
FROM `{DATASET}.mart_channel_quality`
GROUP BY channel_group
ORDER BY users DESC
""")
channels["users"] = channels["users"].astype(int)
channels["purchasers"] = channels["purchasers"].astype(int)
channels["conversion_rate"] = channels["purchasers"] / channels["users"]
channels

,channel_group,users,purchasers,conversion_rate
0,Organic Search,101782,1555,0.0153
1,Unknown,64719,1214,0.0188
2,Direct,64112,1007,0.0157
3,Referral,25250,413,0.0164
4,Paid Search,14291,219,0.0153


### 1.1 İki oran testi (two-proportion z-test)

İki grubun oranlarını karşılaştırmanın standart testi. A/B testlerinde de "B versiyonu A'dan daha mı iyi dönüştü?" sorusu bu testle cevaplanır.

Sadece p-değerine değil, **farkın güven aralığına** da bakıyoruz: fark gerçekte hangi aralıkta olabilir?

In [3]:
by_channel = channels.set_index("channel_group")
counts = np.array([by_channel.loc["Paid Search", "purchasers"], by_channel.loc["Organic Search", "purchasers"]])
nobs = np.array([by_channel.loc["Paid Search", "users"], by_channel.loc["Organic Search", "users"]])

z_stat, p_value = proportions_ztest(counts, nobs)
diff_low, diff_high = confint_proportions_2indep(counts[0], nobs[0], counts[1], nobs[1], method="wald")
diff = counts[0] / nobs[0] - counts[1] / nobs[1]

print(f"Paid Search    : {counts[0]:,} / {nobs[0]:,} = {counts[0] / nobs[0]:.3%}")
print(f"Organic Search : {counts[1]:,} / {nobs[1]:,} = {counts[1] / nobs[1]:.3%}")
print(f"Fark (paid - organic)    : {diff:+.3%}")
print(f"Farkın %95 güven aralığı : [{diff_low:+.3%}, {diff_high:+.3%}]")
print(f"z = {z_stat:.3f}   p = {p_value:.4f}")

Paid Search    : 219 / 14,291 = 1.532%
Organic Search : 1,555 / 101,782 = 1.528%
Fark (paid - organic)    : +0.005%
Farkın %95 güven aralığı : [-0.210%, +0.220%]
z = 0.043   p = 0.9661


### 1.2 Bütün kanallar: ki-kare testi

İki kanaldan fazlasını aynı anda karşılaştırmak için ki-kare bağımsızlık testi: "Dönüşüm, kanala göre değişiyor mu?" Bir kez bütün kanallarla, bir kez de aksiyon alınamayan Unknown hariç bakıyoruz.

In [4]:
def chi_square_by_channel(data: pd.DataFrame) -> None:
    table = np.column_stack([data["purchasers"], data["users"] - data["purchasers"]])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    print(f"Kanallar: {', '.join(data['channel_group'])}")
    print(f"Ki-kare = {chi2:.2f}   serbestlik derecesi = {dof}   p = {p:.4g}\n")


chi_square_by_channel(channels)
chi_square_by_channel(channels[channels["channel_group"] != "Unknown"])

Kanallar: Organic Search, Unknown, Direct, Referral, Paid Search
Ki-kare = 33.24   serbestlik derecesi = 4   p = 1.066e-06

Kanallar: Organic Search, Direct, Referral, Paid Search
Ki-kare = 1.72   serbestlik derecesi = 3   p = 0.6333



### 1.3 Bu veriyle ne büyüklükte bir farkı yakalayabilirdik? (güç analizi)

Fark çıkmaması iki anlama gelebilir: gerçekten fark yoktur, ya da örneklem farkı görmeye yetmeyecek kadar küçüktür. Bunu ayırmak için **en küçük tespit edilebilir etkiyi (MDE)** hesaplıyoruz: α = 0,05 ve %80 güçle, bu örneklem büyüklükleriyle yakalayabileceğimiz en küçük fark.

Aynı hesap bir A/B testi tasarlarken de yapılır: "%10'luk bir artışı yakalamak için grup başına kaç kullanıcı gerekir?"

In [5]:
base_rate = counts[1] / nobs[1]  # organik dönüşüm oranı (referans)
ratio = nobs[1] / nobs[0]        # organik / paid örneklem oranı

h = NormalIndPower().solve_power(
    effect_size=None, nobs1=nobs[0], alpha=0.05, power=0.8, ratio=ratio, alternative="two-sided"
)
# Cohen's h -> oran: h = 2*asin(sqrt(p1)) - 2*asin(sqrt(p2))
detectable_rate = np.sin(np.arcsin(np.sqrt(base_rate)) + h / 2) ** 2

print(f"Referans (organik) dönüşüm oranı           : {base_rate:.3%}")
print(f"%80 güçle ayırt edilebilecek paid dönüşümü : {detectable_rate:.3%}")
print(f"En küçük tespit edilebilir fark (MDE)      : {detectable_rate - base_rate:+.3%} "
      f"(göreli %{(detectable_rate / base_rate - 1) * 100:.0f})")

# A/B testi tasarımı: dönüşümde göreli %10 artışı yakalamak için grup başına gereken kullanıcı
target_rate = base_rate * 1.10
n_per_group = NormalIndPower().solve_power(
    effect_size=proportion_effectsize(target_rate, base_rate), alpha=0.05, power=0.8, ratio=1
)
print(f"\n%{base_rate * 100:.2f} -> %{target_rate * 100:.2f} (göreli +%10) artışı yakalamak için "
      f"grup başına ~{n_per_group:,.0f} kullanıcı gerekir.")

Referans (organik) dönüşüm oranı           : 1.528%
%80 güçle ayırt edilebilecek paid dönüşümü : 1.850%
En küçük tespit edilebilir fark (MDE)      : +0.322% (göreli %21)

%1.53 -> %1.68 (göreli +%10) artışı yakalamak için grup başına ~106,096 kullanıcı gerekir.


### 1.4 Yorum

- **Ücretli ve organik arasında fark yok.** Paid Search %1,532, Organic Search %1,528. Fark +0,005 puan, p = 0,97. Farkın %95 güven aralığı [−0,21, +0,22] puan, yani göreli olarak yaklaşık **±%14**. Gerçek bir fark varsa bile bu aralıktan büyük değil.
- **"Fark yok" sonucu örneklem küçüklüğünden değil.** Bu örneklemlerle göreli %21'lik (+0,32 puan) bir farkı %80 güçle yakalayabilirdik. Bundan büyük bir fark olsaydı görürdük.
- **Kanallar arası ki-kare:** Beş kanal birlikte test edilince fark anlamlı (p ≈ 0,000001), ama bu tamamen **Unknown** kanalından geliyor (%1,88). Bilinen dört kanal arasında fark yok (p = 0,63).
- **A/B testi tasarımı için ders:** %1,53'lük bir dönüşümde göreli %10'luk bir artışı yakalamak için **grup başına ~106.000 kullanıcı** gerekiyor. Düşük dönüşüm oranlarında küçük iyileştirmeleri test etmek büyük trafik ister. Ya testi uzun tutmak ya da funnel'ın daha üst bir adımını (ör. ürün → ödemeye geçme) birincil metrik seçmek gerekir.

**İş sonucu:** Ücretli reklamla gelen kullanıcı, ücretsiz organik kullanıcıdan daha iyi dönüşmüyor. Ücretli reklamın değeri dönüşüm kalitesinde değil, getirdiği **ek** kullanıcı sayısında aranmalı. Bunu ölçmek için incrementality testi (ör. bölgesel kapatma) gerekir.

---
## 2. Soru 6: İlk ziyaretteki hangi davranışlar satın almayı artırıyor?

Veri: `mart_first_session`. Her satır bir yeni kullanıcı (ilk ziyareti 24 Ocak ve öncesi).

- **Hedef:** `purchased_7d`, ilk ziyaretten itibaren 7 gün içinde sipariş verdi mi? Herkese eşit süre tanımak için sabit pencere.
- **Özellikler:** Sadece ilk oturumdan. Ödemeye geçme ve ödeme bilgisi **kullanılmadı**: satın almanın kendisinin parçası oldukları için hedefi sızdırırlar (target leakage).
- **Amaç tahmin değil açıklama:** Hangi davranış, diğerleri sabitken, satın alma olasılığıyla ne kadar ilişkili?

In [6]:
df = run_query(f"""
SELECT
  channel_group,
  device_category,
  period,
  events,
  page_views,
  item_views,
  engagement_time_sec,
  session_duration_sec,
  added_to_cart,
  purchased_first_session,
  purchased_7d
FROM `{DATASET}.mart_first_session`
""")

num_cols = ["events", "page_views", "item_views", "engagement_time_sec", "session_duration_sec"]
df[num_cols] = df[num_cols].astype(float)
df["added_to_cart"] = df["added_to_cart"].astype(int)
df["purchased_7d"] = df["purchased_7d"].astype(int)
df["purchased_first_session"] = df["purchased_first_session"].astype(int)
for col in ["channel_group", "device_category", "period"]:
    df[col] = df[col].astype(str)

print(f"{len(df):,} kullanıcı, 7 gün içinde satın alma oranı: {df['purchased_7d'].mean():.3%}")
df[num_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T

238,649 kullanıcı, 7 gün içinde satın alma oranı: 1.145%


,count,mean,std,min,25%,50%,75%,90%,99%,max
events,"238,649.0000",11.3323,25.7337,2.0000,4.0000,5.0000,9.0000,19.0000,112.0000,726.0000
page_views,"238,649.0000",3.5691,9.1316,0.0000,1.0000,2.0000,3.0000,6.0000,33.0000,411.0000
item_views,"238,649.0000",0.9413,4.6924,0.0000,0.0000,0.0000,0.0000,2.0000,19.0000,176.0000
engagement_time_sec,"238,649.0000",61.9836,189.2279,0.0000,2.3000,9.2000,37.3000,139.7000,918.6120,"9,921.3000"
session_duration_sec,"238,649.0000",168.7695,583.5096,0.0000,4.0000,15.0000,90.0000,473.0000,"2,047.0000","120,362.0000"


### 2.1 Hangi davranış değişkenlerini kullanalım? (korelasyon)

İlk oturumdaki davranışlar birbirine bağlı: çok sayfa gezen biri doğal olarak daha çok olay üretir ve daha uzun kalır. Birbirine çok bağlı değişkenleri aynı modele koymak katsayıları kararsız ve yorumlanamaz yapar (**çoklu doğrusal bağlantı / multicollinearity**). Dağılımlar çarpık olduğu için Spearman (sıra) korelasyonu kullanıyoruz.

In [7]:
df[num_cols].corr(method="spearman").round(2)

,events,page_views,item_views,engagement_time_sec,session_duration_sec
events,1.0000,0.8700,0.5200,0.8300,0.8100
page_views,0.8700,1.0000,0.4700,0.6300,0.6600
item_views,0.5200,0.4700,1.0000,0.4700,0.4400
engagement_time_sec,0.8300,0.6300,0.4700,1.0000,0.8700
session_duration_sec,0.8100,0.6600,0.4400,0.8700,1.0000


**Yorum:** Olay sayısı, sayfa sayısı, etkileşim süresi ve oturum süresi birbirine çok bağlı (0,63 – 0,87). Ürün görüntüleme diğerleriyle daha zayıf bağlı (0,44 – 0,52), yani farklı bir şeyi ölçüyor: ürün ilgisi.

**Karar:** Modelde iki davranış değişkeni tutuldu:
- `item_views`: **ürün ilgisi**
- `engagement_time_sec`: **dikkat / ilgi süresi**. Oturum süresi yerine seçildi çünkü sekme açık bırakılan süreyi değil, aktif etkileşimi ölçüyor.

Olay sayısı, sayfa sayısı ve oturum süresi modele alınmadı, çünkü bunlar etkileşim süresiyle büyük ölçüde aynı bilgiyi taşıyor.

Ayrıca: yeni kullanıcıların **%80'i** ilk oturumunda hiçbir ürüne bakmıyor (medyan 0), ilk oturumların yarısı 9 saniyeden kısa.

### 2.2 Davranışları gruplara ayırmak

Ürün görüntüleme ve etkileşim süresi çok çarpık: çoğu kullanıcı 0 ürüne bakıyor, birkaç kişi onlarca. Sayıyı doğrudan modele koymak "her ek ürün aynı etkiyi yapar" varsayımı demek. Gruplara ayırmak hem bu varsayımı kaldırıyor hem de sonucu iş diline çeviriyor: "4+ ürüne bakanlar, hiç bakmayanlara göre..."

In [8]:
df["item_views_grp"] = pd.cut(
    df["item_views"], bins=[-1, 0, 1, 3, np.inf], labels=["0", "1", "2-3", "4+"]
)
df["engagement_grp"] = pd.cut(
    df["engagement_time_sec"], bins=[-1, 10, 60, 300, np.inf],
    labels=["0-10sn", "10-60sn", "1-5dk", "5dk+"]
)

for col in ["item_views_grp", "engagement_grp", "added_to_cart", "device_category", "channel_group", "period"]:
    summary = df.groupby(col, observed=True)["purchased_7d"].agg(users="size", purchasers="sum", rate="mean")
    print(f"\n=== {col} ===")
    print(summary.to_string(formatters={"rate": "{:.3%}".format}))


=== item_views_grp ===
                 users  purchasers    rate
item_views_grp                            
0               191884         307  0.160%
1                21072         262  1.243%
2-3              13161         408  3.100%
4+               12532        1756 14.012%

=== engagement_grp ===
                 users  purchasers    rate
engagement_grp                            
0-10sn          123820          86  0.069%
10-60sn          69919         165  0.236%
1-5dk            33359         621  1.862%
5dk+             11551        1861 16.111%

=== added_to_cart ===
                users  purchasers    rate
added_to_cart                            
0              230517        1390  0.603%
1                8132        1343 16.515%

=== device_category ===
                  users  purchasers   rate
device_category                           
desktop          138626        1566 1.130%
mobile            94672        1114 1.177%
tablet             5351          53 0.990%

=== 

### 2.3 Lojistik regresyon

Her kategorik değişken bir **referans grubuna** göre yorumlanır:

| Değişken | Referans |
|---|---|
| Ürün görüntüleme | 0 ürün |
| Etkileşim süresi | 0-10 sn |
| Cihaz | masaüstü |
| Kanal | Organic Search |
| Dönem | Kasım (Black Friday öncesi) |

Dönem modele **kontrol değişkeni** olarak eklendi: trend analizinde dönemler arasında büyük fark vardı. Kontrol edilmezse davranışların etkisine dönem etkisi karışabilir.

In [9]:
formula = (
    "purchased_7d ~ C(item_views_grp, Treatment('0'))"
    " + C(engagement_grp, Treatment('0-10sn'))"
    " + added_to_cart"
    " + C(device_category, Treatment('desktop'))"
    " + C(channel_group, Treatment('Organic Search'))"
    " + C(period, Treatment('1_kasim'))"
)
model = smf.logit(formula, data=df).fit(disp=False)
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:           purchased_7d   No. Observations:               238649
Model:                          Logit   Df Residuals:                   238632
Method:                           MLE   Df Model:                           16
Date:                Mon, 23 Nov 2026   Pseudo R-squ.:                  0.3754
Time:                        22:31:30   Log-Likelihood:                -9326.7
converged:                       True   LL-Null:                       -14933.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------------------
Intercept                                                       -7.3889      0.123    -60.214      0.000      

### 2.4 Odds ratio'lar

Lojistik regresyon katsayıları log-odds cinsinden. `exp(katsayı)` = **odds ratio**: diğer her şey sabitken, o grubun satın alma oddsu referans grubun kaç katı?

- **1'den büyük:** Satın alma ihtimali daha yüksek.
- **1'den küçük:** Daha düşük.
- **Güven aralığı 1'i içeriyorsa** fark istatistiksel olarak anlamlı değil.

In [10]:
ci = model.conf_int()
odds = pd.DataFrame({
    "odds_ratio": np.exp(model.params),
    "ci_low": np.exp(ci[0]),
    "ci_high": np.exp(ci[1]),
    "p_value": model.pvalues,
}).drop(index="Intercept")
odds.index = (
    odds.index.str.replace(r"C\((\w+), Treatment\('[^']*'\)\)\[T\.", r"\1 = ", regex=True)
    .str.rstrip("]")
)
odds.round(3)

,odds_ratio,ci_low,ci_high,p_value
item_views_grp = 1,2.8760,2.4060,3.4380,0.0000
item_views_grp = 2-3,3.1860,2.6740,3.7970,0.0000
item_views_grp = 4+,4.7870,4.0300,5.6850,0.0000
engagement_grp = 10-60sn,2.5800,1.9810,3.3600,0.0000
engagement_grp = 1-5dk,10.1380,7.9050,13.0010,0.0000
engagement_grp = 5dk+,46.2940,35.7900,59.8820,0.0000
device_category = mobile,1.0660,0.9800,1.1590,0.1380
device_category = tablet,0.8700,0.6460,1.1720,0.3600
channel_group = Direct,0.9610,0.8630,1.0690,0.4620
channel_group = Paid Search,1.1170,0.9300,1.3410,0.2360


### 2.5 Olasılık puanı olarak etki (ortalama marjinal etki)

Taban oran %1,15 gibi küçük olduğunda odds ratio'lar büyük görünür. "Yüzde kaç puan artırıyor?" sorusu iş tarafı için daha anlaşılır. Ortalama marjinal etki, her değişkenin 7 günlük satın alma olasılığını ortalamada kaç **yüzde puan** değiştirdiğini gösterir.

In [11]:
margeff = model.get_margeff(at="overall", dummy=True).summary_frame()
margeff.index = odds.index
(margeff.iloc[:, [0, 4, 5]] * 100).set_axis(["etki_puan", "ci_low", "ci_high"], axis=1).round(2)

,etki_puan,ci_low,ci_high
item_views_grp = 1,1.3500,1.0600,1.6400
item_views_grp = 2-3,1.4700,1.1900,1.7600
item_views_grp = 4+,1.8400,1.5600,2.1100
engagement_grp = 10-60sn,1.2000,0.7800,1.6200
engagement_grp = 1-5dk,3.2600,2.7900,3.7300
engagement_grp = 5dk+,10.2300,8.5800,11.8700
device_category = mobile,0.0600,-0.0200,0.1400
device_category = tablet,-0.1300,-0.3900,0.1300
channel_group = Direct,-0.0400,-0.1400,0.0600
channel_group = Paid Search,0.1100,-0.0800,0.3000


### 2.6 Model ne kadar iyi ayırt ediyor?

Amaç açıklama olduğu için bunlar sadece sağlama. Sınıflar çok dengesiz (%1,15 pozitif); "kimse almaz" diyen bir model bile %98,85 doğru olurdu. Bu yüzden doğruluk (accuracy) değil, **ROC AUC** (alıcıları alıcı olmayanlardan ne kadar iyi sıraladığı; 0,5 = yazı tura) ve **McFadden pseudo R²** kullanıyoruz.

In [12]:
df["p_hat"] = model.predict(df)
pos = df["purchased_7d"] == 1
ranks = df["p_hat"].rank()
auc = (ranks[pos].sum() - pos.sum() * (pos.sum() + 1) / 2) / (pos.sum() * (~pos).sum())

print(f"McFadden pseudo R² : {model.prsquared:.3f}")
print(f"ROC AUC            : {auc:.3f}")

McFadden pseudo R² : 0.375
ROC AUC            : 0.937


### 2.7 Sağlamlık kontrolü: ilk oturumda satın alanları çıkarmak

İlk modelin açıklama gücü şüpheli derecede yüksek (pseudo R² 0,375, AUC 0,937). Olası neden: 7 gün içinde satın alanların bir kısmı **satın almayı ilk oturumun içinde** yapmış. Onların etkileşim süresi, ürün görüntülemesi ve sepete eklemesi satın alma sürecinin kendisini de içeriyor. Ödeme yapan biri doğal olarak daha uzun kalır. Bu, hedefin özelliklere kısmen sızması demek.

Kontrol: İlk oturumda satın alanları çıkarıp modeli tekrar kuruyoruz. Soru artık şu: **"İlk oturumda satın almayan bir kullanıcının ilk oturum davranışı, sonraki günlerde satın alacağını gösteriyor mu?"** Bu, pazarlama için daha kullanışlı bir soru: kimi yeniden hedeflemeli (retargeting)?

In [13]:
def roc_auc(y_true: pd.Series, score: pd.Series) -> float:
    pos = y_true == 1
    ranks = score.rank()
    return (ranks[pos].sum() - pos.sum() * (pos.sum() + 1) / 2) / (pos.sum() * (~pos).sum())


buyers_7d = df["purchased_7d"].sum()
first_session_buyers = ((df["purchased_first_session"] == 1) & (df["purchased_7d"] == 1)).sum()
inconsistent = ((df["purchased_first_session"] == 1) & (df["purchased_7d"] == 0)).sum()
print(f"7 gün içinde satın alan          : {buyers_7d:,}")
print(f"  bunlardan ilk oturumda alan    : {first_session_buyers:,} (%{100 * first_session_buyers / buyers_7d:.1f})")
print(f"  ilk oturumdan sonra alan       : {buyers_7d - first_session_buyers:,}")
print(f"Tutarsız (ilk oturumda almış ama 7 günlük hedef 0): {inconsistent:,}  (beklenen 0, bkz. mart_first_session_checks)")

later = df[df["purchased_first_session"] == 0].copy()
model_later = smf.logit(formula, data=later).fit(disp=False)
ci_later = model_later.conf_int()
odds_later = pd.DataFrame({
    "odds_ratio": np.exp(model_later.params),
    "ci_low": np.exp(ci_later[0]),
    "ci_high": np.exp(ci_later[1]),
    "p_value": model_later.pvalues,
}).drop(index="Intercept")
odds_later.index = odds.index

print(f"\nAlt örneklem: {len(later):,} kullanıcı, sonradan satın alma oranı: {later['purchased_7d'].mean():.3%}")
print(f"McFadden pseudo R² : {model_later.prsquared:.3f}   (ilk model {model.prsquared:.3f})")
print(f"ROC AUC            : {roc_auc(later['purchased_7d'], model_later.predict(later)):.3f}   (ilk model {auc:.3f})")

comparison = pd.DataFrame({
    "ilk_model_OR": odds["odds_ratio"],
    "sonraki_alıcılar_OR": odds_later["odds_ratio"],
    "ci_low": odds_later["ci_low"],
    "ci_high": odds_later["ci_high"],
    "p_value": odds_later["p_value"],
})
comparison.to_csv(Path("../data") / "logit_odds_ratios_robustness.csv")
comparison.round(3)

7 gün içinde satın alan          : 2,733
  bunlardan ilk oturumda alan    : 1,642 (%60.1)
  ilk oturumdan sonra alan       : 1,091

Alt örneklem: 237,007 kullanıcı, sonradan satın alma oranı: 0.461%
McFadden pseudo R² : 0.193   (ilk model 0.375)
ROC AUC            : 0.873   (ilk model 0.937)


,ilk_model_OR,sonraki_alıcılar_OR,ci_low,ci_high,p_value
item_views_grp = 1,2.8760,1.9370,1.5490,2.4220,0.0000
item_views_grp = 2-3,3.1860,2.4280,1.9490,3.0250,0.0000
item_views_grp = 4+,4.7870,4.6840,3.7650,5.8280,0.0000
engagement_grp = 10-60sn,2.5800,2.9250,2.2470,3.8080,0.0000
engagement_grp = 1-5dk,10.1380,8.6590,6.6560,11.2640,0.0000
engagement_grp = 5dk+,46.2940,14.8390,11.0490,19.9300,0.0000
device_category = mobile,1.0660,1.0390,0.9180,1.1750,0.5490
device_category = tablet,0.8700,0.8650,0.5560,1.3450,0.5200
channel_group = Direct,0.9610,0.8330,0.7110,0.9770,0.0250
channel_group = Paid Search,1.1170,1.0070,0.7700,1.3160,0.9620


### 2.8 Kimi yeniden hedeflemeli? (retargeting kitlesi)

İlk oturumda satın almayan ~237 bin yeni kullanıcının sadece **%0,46**'sı, 7 günlük pencere içinde sonradan satın alıyor. Reklam bütçesi sınırlı: herkesi değil, geri dönüp alma ihtimali yüksek olanları hedeflemek gerekir. İki yoldan bakıyoruz:

1. **Davranış matrisi ve kural bazlı kitle:** Reklam platformlarında (GA4 / Google Ads kitleleri) model olmadan doğrudan kurulabilecek basit bir tanım: ilk oturumda **1 dakikadan fazla etkileşim** ya da **4+ ürün görüntüleme**.
2. **Model skoruna göre dilimler (kazanç tablosu / gain table):** Kullanıcıları modelin tahmin ettiği olasılığa göre 10 eşit dilime ayırıp en üst dilimlerin, sonradan alanların ne kadarını yakaladığına bakıyoruz. Pazarlamada "kitlenin en iyi %10'unu hedeflersek alıcıların yüzde kaçına ulaşırız?" sorusunun cevabı.

In [14]:
base_later = later["purchased_7d"].mean()

# 1) Etkileşim süresi × ürün görüntüleme: sonradan satın alma oranı (kullanıcı sayısı)
cells_ = later.groupby(["engagement_grp", "item_views_grp"], observed=True)["purchased_7d"].agg(["mean", "size"])
matrix = cells_.apply(lambda r: f"{r['mean']:.2%} ({int(r['size']):,})", axis=1).unstack()
print("Sonradan satın alma oranı (kullanıcı sayısı) | satır: etkileşim, sütun: ürün görüntüleme")
display(matrix)

# 2) Kural bazlı kitle: 1 dk+ etkileşim ya da 4+ ürün görüntüleme
hot = later["engagement_grp"].isin(["1-5dk", "5dk+"]) | (later["item_views_grp"] == "4+")
audience = later.groupby(hot.map({True: "sıcak kitle", False: "diğer"}))["purchased_7d"].agg(users="size", buyers="sum")
audience["user_share"] = audience["users"] / audience["users"].sum()
audience["buyer_share"] = audience["buyers"] / audience["buyers"].sum()
audience["rate"] = audience["buyers"] / audience["users"]
audience["lift"] = audience["rate"] / base_later
print("\nKural bazlı kitle")
display(audience.round(4))

# 3) Model skoruna göre 10 dilim (1 = en yüksek tahmin edilen olasılık)
scored = later.assign(score=model_later.predict(later))
scored["decile"] = pd.qcut(scored["score"].rank(method="first", ascending=False), 10, labels=range(1, 11))
gain = scored.groupby("decile", observed=True)["purchased_7d"].agg(users="size", buyers="sum")
gain["rate"] = gain["buyers"] / gain["users"]
gain["lift"] = gain["rate"] / base_later
gain["cum_buyer_share"] = gain["buyers"].cumsum() / gain["buyers"].sum()
gain.to_csv(Path("../data") / "retargeting_gain_table.csv")
print("\nKazanç tablosu (model skoruna göre 10 dilim)")
gain.round(4)

Sonradan satın alma oranı (kullanıcı sayısı) | satır: etkileşim, sütun: ürün görüntüleme


item_views_grp,0,1,2-3,4+
engagement_grp,,,,
0-10sn,"0.06% (116,264)","0.16% (6,852)",0.47% (639),1.54% (65)
10-60sn,"0.18% (58,564)","0.37% (7,629)","0.65% (3,363)",1.93% (362)
1-5dk,"0.63% (15,609)","1.19% (5,706)","1.59% (7,418)","2.74% (4,410)"
5dk+,"0.91% (1,430)",2.01% (745),"1.80% (1,503)","5.91% (6,448)"



Kural bazlı kitle


,users,buyers,user_share,buyer_share,rate,lift
diğer,193311,242,0.8156,0.2216,0.0013,0.2717
sıcak kitle,43696,850,0.1844,0.7784,0.0195,4.2220



Kazanç tablosu (model skoruna göre 10 dilim)


,users,buyers,rate,lift,cum_buyer_share
decile,,,,,
1,23701,710,0.0300,6.5017,0.6502
2,23701,169,0.0071,1.5476,0.8049
3,23700,68,0.0029,0.6227,0.8672
4,23701,47,0.0020,0.4304,0.9103
5,23701,25,0.0011,0.2289,0.9332
6,23700,24,0.0010,0.2198,0.9551
7,23701,13,0.0005,0.1190,0.9670
8,23700,19,0.0008,0.1740,0.9844
9,23701,11,0.0005,0.1007,0.9945


**Yorum:**
- **İki boyut birlikte artıyor.** Hem süre hem ürün ilgisi arttıkça sonradan satın alma oranı yükseliyor. En iyi hücre (5 dk+ ve 4+ ürün): 6.448 kullanıcı, **%5,91**, ortalamanın (%0,46) ~13 katı. En kötü hücre (0-10 sn ve 0 ürün): 116.264 kullanıcı (ilk oturumda almayanların **%49'u**), **%0,06**.
- **Basit kural iyi çalışıyor:** "1 dk+ etkileşim ya da 4+ ürün görüntüleme" kuralı, ilk oturumda almayanların **%18'ini** seçiyor ve sonradan alanların **%78'ini** yakalıyor. Bu kitlenin satın alma oranı %1,95 (lift 4,2), geri kalanınki %0,13.
- **Model biraz daha keskin:** En yüksek skorlu %10 (23.701 kullanıcı), sonradan alanların **%65'ini** yakalıyor (oran %3,0, lift 6,5). İlk %20 alıcıların %80'ini yakalıyor. Alt %50 ise alıcıların sadece %7'sini içeriyor.
- **Kural ile model neredeyse aynı:** Kural kitlenin %18'iyle alıcıların %78'ini, modelin ilk %20'si %80'ini yakalıyor. Yani reklam platformunda model kurmadan, basit bir davranış kuralıyla neredeyse aynı kitle kurulabilir. Bütçe daha da sınırlıysa modelin ilk %10'u kullanılabilir.
- **Bütçe karşılığı:** İlk oturumda 10 saniyeden kısa kalıp hiçbir ürüne bakmayanlar (kitlenin yarısı) hedeflenmezse, yeniden hedefleme bütçesinin yaklaşık yarısı kurtarılır; kaçırılan alıcı ~70 kişi (~%6).

**Dikkat (artımsal etki):** Bu kitle, sonradan satın alma ihtimali en yüksek olanlar. Ama bir kısmı reklam görmeden de satın alacaktı. Yüksek skorlu kitleyi hedeflemek, reklamın satışı artırdığı anlamına gelmez. Kitlenin bir kısmı reklamsız bırakılarak (**holdout grubu**) reklamın artımsal etkisi ölçülmeli. Bir sonraki adım, "kim alır?" yerine "kim reklam sayesinde alır?" sorusunu soran **uplift modellemesi**.

### 2.9 Yorum ve sınırlar

**1. Sızıntı şüphesi doğrulandı.** 7 gün içinde satın alan 2.733 yeni kullanıcının **%60'ı (1.641) satın almayı ilk oturumunda yapmış.** Bu kullanıcılar çıkarılınca modelin açıklama gücü yaklaşık yarıya iniyor (pseudo R² 0,375 → 0,193, AUC 0,937 → 0,873). İlk modelin gücünün bir kısmı "ödeme yapan kişi sitede uzun kalır" gibi bariz bir ilişkiden geliyordu. Yine de AUC 0,87: ilk oturum davranışı, sonradan kimin alacağını gerçekten ayırt ediyor.

**2. İki model yan yana (odds ratio):**

| Değişken | İlk model | Sonradan alanlar | Yorum |
|---|---|---|---|
| 4+ ürün görüntüleme | 4,8 | 4,7 | Neredeyse hiç değişmedi: **en sağlam sinyal** |
| Etkileşim 5 dk+ | 46,3 | 14,8 | Hâlâ en güçlü, ama etkinin büyük kısmı aynı oturumdaki ödeme süresiydi |
| Etkileşim 1-5 dk | 10,1 | 8,7 | Az değişti |
| Sepete ekleme | 3,1 | 1,7 | Yarıya indi: etkisinin çoğu aynı oturumda satın almaktan geliyordu |
| Cihaz | ≈ 1 | ≈ 1 | İki modelde de etkisiz |
| Kanal | ≈ 1 | 0,83 – 1,01 | Direct 0,83 (p = 0,025), ama 4 kanal karşılaştırıldığı için Bonferroni düzeltmesiyle (0,05 / 4 = 0,0125) anlamlı değil |
| Sezon dönemi (Kasım'a göre) | 1,15 | 1,73 | Black Friday – Aralık ortası gelenler ilk oturumda almasa da sonradan daha çok alıyor |
| Tatil dönemi (Kasım'a göre) | 0,65 | 0,62 | İki modelde de düşük |

**3. İş sonucu:**
- **Yeni kullanıcı dönüşümünün çoğu (%60) ilk oturumda oluyor.** İlk oturum deneyimi en kritik nokta.
- İlk oturumda almayanlar arasında kimin sonradan alacağını **nereden geldiği değil, ilk oturumda ne yaptığı** gösteriyor: kaç ürüne baktığı ve ne kadar kaldığı. Bu, Soru 5'in sonucuyla tutarlı: kanallar arasında dönüşüm farkı yok.
- Ürün ilgisi ve süre hesaba katıldığında **sepete eklemenin ek bilgisi az** (OR 1,7). Klasik "sepeti terk edenler" kitlesi, ilgili kullanıcıların sadece bir alt kümesi.
- Bu yüzden yeniden hedefleme kitlesi kanal ya da sepet bazında değil, **ilk oturum davranışına göre** kurulmalı (2.8).

**Sınırlar:**
- **İlişki, nedensellik değil.** Kullanıcıyı daha uzun tutmak ya da daha çok ürüne baktırmak satışı artırmayabilir; zaten almaya niyetli kullanıcı daha çok bakıyor olabilir. Bulgular **kimi hedefleyeceğimizi** söylüyor, neyi değiştirirsek satışın artacağını söylemiyor. Bunun için A/B testi gerekir.
- Etkileşim süresi, ödeme sayfasına geçip vazgeçenlerin orada geçirdiği süreyi de içeriyor olabilir. Ödemeye geçme modele alınmadı ama süre üzerinden dolaylı girebilir.
- Yeniden hedefleme kitlesi "kim alır?" sorusunun cevabı, "kim reklam sayesinde alır?" sorusunun değil (2.8, artımsal etki).
- Kazanç tablosu modelin kurulduğu veriyle hesaplandı. 16 değişken / 237 bin satır olduğu için aşırı uyum (overfitting) riski düşük; gerçek kullanımda model bir dönemde kurulup sonraki dönemde test edilmeli.
- 7 günlük pencereden sonra alanlar "almadı" sayıldı. Farklı bir pencere sonuçları değiştirebilir.
- Kanal ve cihaz etiketleri anonimleştirme yüzünden kısmen karışmış olabilir (veri kalitesi notu ve 5.4).

---
## 3. Tableau için CSV çıktıları

Tableau Public doğrudan BigQuery'ye bağlanamadığı için mart tabloları `data/` klasörüne CSV olarak yazılıyor. Odds ratio tablosu da README ve dashboard için kaydediliyor.

In [15]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

for table in ["mart_funnel", "mart_channel_quality", "mart_cohort_retention", "mart_daily_kpis"]:
    out = run_query(f"SELECT * FROM `{DATASET}.{table}`")
    out.to_csv(DATA_DIR / f"{table}.csv", index=False)
    print(f"{table:<24} {len(out):>6,} satır -> data/{table}.csv")

odds.to_csv(DATA_DIR / "logit_odds_ratios.csv")
print(f"{'logit_odds_ratios':<24} {len(odds):>6,} satır -> data/logit_odds_ratios.csv")

mart_funnel                  15 satır -> data/mart_funnel.csv
mart_channel_quality         15 satır -> data/mart_channel_quality.csv
mart_cohort_retention       525 satır -> data/mart_cohort_retention.csv
mart_daily_kpis             460 satır -> data/mart_daily_kpis.csv
logit_odds_ratios            16 satır -> data/logit_odds_ratios.csv


Dashboard için iki modelin odds ratio'ları tek tabloda, **uzun formatta** (her satır bir model × değişken) ve İngilizce etiketlerle yazılıyor. Tableau'da iki modeli yan yana çizmek (forest plot) bu formatta tek bir grafikle yapılabiliyor. `significant` bayrağı Bonferroni düzeltmeli: eşik, değişken grubundaki karşılaştırma sayısına bölünüyor (6.9; ör. 4 kanal için 0,05 / 4).

In [ ]:
TERMS = {
    "item_views_grp = 1":          ("Item views (vs 0)", "1 item"),
    "item_views_grp = 2-3":        ("Item views (vs 0)", "2-3 items"),
    "item_views_grp = 4+":         ("Item views (vs 0)", "4+ items"),
    "engagement_grp = 10-60sn":    ("Engagement time (vs 0-10s)", "10-60 s"),
    "engagement_grp = 1-5dk":      ("Engagement time (vs 0-10s)", "1-5 min"),
    "engagement_grp = 5dk+":       ("Engagement time (vs 0-10s)", "5+ min"),
    "added_to_cart":               ("Cart", "Added to cart"),
    "device_category = mobile":    ("Device (vs desktop)", "Mobile"),
    "device_category = tablet":    ("Device (vs desktop)", "Tablet"),
    "channel_group = Direct":      ("Channel (vs Organic Search)", "Direct"),
    "channel_group = Paid Search": ("Channel (vs Organic Search)", "Paid Search"),
    "channel_group = Referral":    ("Channel (vs Organic Search)", "Referral"),
    "channel_group = Unknown":     ("Channel (vs Organic Search)", "Unknown"),
    "period = 2_sezon":            ("Period (vs early Nov)", "Peak season (Nov 23 - Dec 20)"),
    "period = 3_tatil":            ("Period (vs early Nov)", "Holidays (Dec 21 - Jan 3)"),
    "period = 4_ocak":             ("Period (vs early Nov)", "January"),
}


def odds_long(table: pd.DataFrame, model_name: str) -> pd.DataFrame:
    out = table[["odds_ratio", "ci_low", "ci_high", "p_value"]].rename_axis("term").reset_index()
    out["model"] = model_name
    out["variable_group"] = out["term"].map(lambda t: TERMS[t][0])
    out["level"] = out["term"].map(lambda t: TERMS[t][1])
    # Bonferroni: grup içindeki karşılaştırma sayısına göre eşik (ör. 4 kanal -> 0,05 / 4)
    out["significant"] = out["p_value"] < 0.05 / out.groupby("variable_group")["term"].transform("size")
    return out


tableau_odds = pd.concat([
    odds_long(odds, "All new users"),
    odds_long(odds_later, "First-session non-buyers"),
], ignore_index=True)
tableau_odds.to_csv(DATA_DIR / "tableau_odds_ratios.csv", index=False)
print(f"{'tableau_odds_ratios':<24} {len(tableau_odds):>6,} satır -> data/tableau_odds_ratios.csv")